***Dependencies***

In [ ]:
# import sys
# !{sys.executable} -m pip install MissForest

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import time

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from missforest import MissForest

from config import get_random_num, count_missing, CATEGORICAL_FEATURES, SAMPLE_NUM

rng_num = get_random_num()

INPUT_DATA_PATH = "missing_train_splits"

def load_data_from_folder(folder_path):
    folder = Path(folder_path)
    df_list = []
    file_list = []
    for file in folder.glob("*.csv"):
        df_list.append(pd.read_csv(file))
        file_list.append(file)
    return file_list, df_list


***Defining imputation methods***

In [2]:
def impute_mean_mode(df, cat_features):
    num_features = [col for col in df.columns if col not in cat_features]
    transformer = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='mean'), num_features),
            ('cat', SimpleImputer(strategy='most_frequent'), cat_features)
        ]
    )
    df_imputed = pd.DataFrame(transformer.fit_transform(df), columns=num_features + cat_features)
    df_imputed = df_imputed[df.columns]

    for col in cat_features:
        df_imputed[col] = df_imputed[col].round().astype(int)

    return df_imputed

def impute_median_mode(df, cat_features):
    num_features = [col for col in df.columns if col not in cat_features]
    transformer = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), num_features),
            ('cat', SimpleImputer(strategy='most_frequent'), cat_features)
        ]
    )
    df_imputed = pd.DataFrame(transformer.fit_transform(df), columns=num_features + cat_features)
    df_imputed = df_imputed[df.columns]
    
    for col in cat_features:
        df_imputed[col] = df_imputed[col].round().astype(int)
    
    return df_imputed

In [ ]:
def impute_knn(df, cat_features, n_neighbors=5):
    df_encoded = df.copy()

    ohe_maps = {}
    for col in cat_features:
        if col in df_encoded.columns:
            dummies = pd.get_dummies(df_encoded[col], prefix=col, dummy_na=False, dtype=float)
            
            null_mask = df_encoded[col].isna()
            if null_mask.any():
                dummies[null_mask] = np.nan
                
            ohe_maps[col] = dummies.columns.tolist()
            df_encoded = df_encoded.drop(columns=[col]).join(dummies)

    scaler = MinMaxScaler()
    scaled_array = scaler.fit_transform(df_encoded)

    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed_array = imputer.fit_transform(scaled_array)

    df_imputed = pd.DataFrame(scaler.inverse_transform(imputed_array), columns=df_encoded.columns)

    for col, dummy_cols in ohe_maps.items():
        max_dummy = df_imputed[dummy_cols].idxmax(axis=1)
        cat_values = max_dummy.str.split(f"{col}_", n=1).str[1]
        df_imputed[col] = cat_values.astype(float).astype(int)
        df_imputed = df_imputed.drop(columns=dummy_cols)

    return df_imputed[df.columns]

def impute_hybrid_regression(df, cat_features):
    df_imp = df.copy()

    for col in df_imp.columns:
        if col in cat_features:
            mode_val = df_imp[col].mode()
            df_imp[col] = df_imp[col].fillna(mode_val.iloc[0] if not mode_val.empty else 0)
        else:
            median_val = df_imp[col].median()
            df_imp[col] = df_imp[col].fillna(median_val if not np.isnan(median_val) else 0)
            
    for target_col in df.columns:
        null_mask = df[target_col].isna()
        if not null_mask.any():
            continue
            
        X = df_imp.drop(columns=[target_col])
        y = df_imp[target_col]

        X = pd.get_dummies(X, columns=[c for c in cat_features if c != target_col], drop_first=True)

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        X_train = X_scaled[~null_mask]
        y_train = y[~null_mask]
        X_test = X_scaled[null_mask]
        
        if target_col in cat_features:
            y_train = y_train.astype(int)
            n_classes = len(np.unique(y_train))

            if n_classes <= 1:
                predictions = np.full(X_test.shape[0], y_train.iloc[0])
            else:
                solver = 'liblinear' if n_classes == 2 else 'lbfgs'
                model = LogisticRegression(max_iter=500, solver=solver)
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)
        else:
            model = LinearRegression()
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            
        df_imp.loc[null_mask, target_col] = predictions

    for col in cat_features:
        if col in df_imp.columns:
            df_imp[col] = df_imp[col].round().astype(int)
            
    return df_imp


In [ ]:
def impute_MICE(df, cat_features):
    df_encoded = df.copy()

    ohe_maps = {}
    for col in cat_features:
        if col in df_encoded.columns:
            dummies = pd.get_dummies(df_encoded[col], prefix=col, dummy_na=False, dtype=float)
            null_mask = df_encoded[col].isna()
            if null_mask.any():
                dummies[null_mask] = np.nan
            ohe_maps[col] = dummies.columns.tolist()
            df_encoded = df_encoded.drop(columns=[col]).join(dummies)

    imputer = IterativeImputer(
        estimator=HistGradientBoostingRegressor(max_depth=6, learning_rate=0.1, random_state=rng_num),
        max_iter=10,
        random_state=rng_num,
        skip_complete=True
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        imputed_array = imputer.fit_transform(df_encoded)
        
    df_imputed = pd.DataFrame(imputed_array, columns=df_encoded.columns)

    for col, dummy_cols in ohe_maps.items():
        max_dummy = df_imputed[dummy_cols].idxmax(axis=1)
        cat_values = max_dummy.str.split(f"{col}_", n=1).str[1]
        df_imputed[col] = cat_values.astype(float).astype(int)
        df_imputed = df_imputed.drop(columns=dummy_cols)

    return df_imputed[df.columns]

def impute_missforest(df, cat_features):
    df_imp = df.copy()

    cat_cols = [col for col in cat_features if col in df_imp.columns]

    imputer = MissForest(
        max_iter=10,
        categorical=cat_cols)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        df_imputed_array = imputer.fit_transform(df_imp)

    df_imputed = pd.DataFrame(df_imputed_array, columns=df.columns)

    for col in cat_cols:
        df_imputed[col] = df_imputed[col].round().astype(int)

    return df_imputed

***Imputing training sets with missing values***

In [ ]:
imputations = {
        "mean": impute_mean_mode,
        "median": impute_median_mode,
        "knn": impute_knn,
        "regression": impute_hybrid_regression,
        "mice": impute_MICE,
        "missforest": impute_missforest
    }

for i in range(SAMPLE_NUM):
    file_list, df_list = load_data_from_folder(INPUT_DATA_PATH + f"/sample{i+1}")
    OUTPUT_DATA_FOLDER = Path(f"imputed_train_splits/sample{i+1}")
    OUTPUT_DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    OUTPUT_REPORTS_FOLDER = Path(f"imputation_reports/sample{i+1}")
    OUTPUT_REPORTS_FOLDER.mkdir(exist_ok=True, parents=True)

    for path, df in zip(file_list, df_list):
        file_name = path.stem
        print(f"Procesing: Sample {i+1} - {file_name}")

        disease_name = file_name.split('_train')[0]
        current_categorical_features = CATEGORICAL_FEATURES.get(disease_name, [])

        for method_name, imp_func in imputations.items():
            try:
                total, missing, percent_missing = count_missing(df)
                start_time = time.time()
                df_imp = imp_func(df.copy(), current_categorical_features)
                end_time = time.time()
                elapsed_time = end_time - start_time

                out_path_data = OUTPUT_DATA_FOLDER / f"{file_name}_{method_name}.csv"
                df_imp.to_csv(out_path_data, index=False)
                print(f" -> Saved: {out_path_data}")

                out_path_time = OUTPUT_REPORTS_FOLDER / f"{file_name}_{method_name}.txt"
                with open(out_path_time, "w", encoding="utf-8") as f:
                    f.write(f"Total cells: {total}\n")
                    f.write(f"Missing cells: {missing}\n")
                    f.write(f"Percent missing: {percent_missing}\n")
                    f.write(f"Elapsed time: {elapsed_time:.14f} s\n")
                    f.write(f"Average time per cell: {(elapsed_time / total):.14f} s\n")
                print(f" -> Saved: {out_path_time}")
            except Exception as e:
                print(f" -> Error occurred while processing {method_name}: {e}")

        print("\n")
